## Sentinel-2 Post-Processing Script
==================================

Steps:
  1. Reprojecting all bands from EPSG:4326 → EPSG:32647 at 10m
  2. Resampling Band 11 from 20m → 10m (bilinear)
  3. Calculating 9 spectral indices
  4. Saving stacked multi-band GeoTIFF (bands + indices) in EPSG:32647

Input : Sentinel2_2024_Kachin.tif  (bands: B2, B3, B4, B8, B11)

Output: Sentinel2_2024_Kachin_processed.tif

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling as ResamplingEnum
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 0. Configuration
INPUT_TIF  = "../region_of_interest/Sentinel2_2025_Kachin.tif"
OUTPUT_TIF = "Sentinel2_2025_Kachin_processed.tif"
TARGET_CRS       = "EPSG:32647"
TARGET_RESOLUTION = 10  # meters

# Band order in the exported GeoTIFF (from GEE export)
# B2=1, B3=2, B4=3, B8=4, B11=5
BAND_IDX = {
    'B2' : 1,   # Blue
    'B3' : 2,   # Green
    'B4' : 3,   # Red
    'B8' : 4,   # NIR
    'B11': 5,   # SWIR (20m — will be resampled)
}

Step 1: Reprojecting all bands to EPSG:32647 at 10m ...
  Reprojected to EPSG:32647 | size: 2281 x 1783 pixels


In [ ]:
# 1. REPROJECT all bands to EPSG:32647 at 10m
print("Step 1: Reprojecting all bands to EPSG:32647 at 10m ...")

with rasterio.open(INPUT_TIF) as src:
    src_crs      = src.crs
    src_transform = src.transform
    src_count    = src.count  # number of bands
    src_dtype    = src.dtypes[0]
    src_nodata   = src.nodata

    # Calculate transform for target CRS at 10m resolution
    transform, width, height = calculate_default_transform(   #using rasterio library
        src_crs,
        TARGET_CRS,
        src.width,
        src.height,
        *src.bounds,
        resolution=TARGET_RESOLUTION
    )

    reprojected_profile = src.meta.copy()
    reprojected_profile.update({
        'crs'      : TARGET_CRS,
        'transform': transform,
        'width'    : width,
        'height'   : height,
        'dtype'    : 'float32',
        'nodata'   : np.nan,
    })

    # Reproject all bands into memory
    reprojected_bands = np.full(
        (src_count, height, width), np.nan, dtype=np.float32
    )

    for band_num in range(1, src_count + 1):
        reproject(
            source      =rasterio.band(src, band_num),
            destination =reprojected_bands[band_num - 1],
            src_transform=src_transform,
            src_crs      =src_crs,
            dst_transform=transform,
            dst_crs      =TARGET_CRS,           #target_CRS
            resampling   =Resampling.bilinear,  #bilinear (2x2) resampling
            src_nodata   =src_nodata,
            dst_nodata   =np.nan,
        )

print(f"  Reprojected to {TARGET_CRS} | size: {width} x {height} pixels")

In [ ]:
# 2. EXTRACT individual bands (already at 10m after reproject)
#    Band 11 was 20m in original but reproject already brought it to 10m grid.
#    We apply explicit bilinear resample to ensure spatial accuracy.

print("Step 2: Extracting bands (B11 confirmed at 10m via bilinear) ...")

# Scale factor: GEE exports S2 SR values as integers (divide by 10000)
SCALE = 10000.0

B2  = reprojected_bands[BAND_IDX['B2']  - 1] / SCALE   # Blue
B3  = reprojected_bands[BAND_IDX['B3']  - 1] / SCALE   # Green
B4  = reprojected_bands[BAND_IDX['B4']  - 1] / SCALE   # Red
B8  = reprojected_bands[BAND_IDX['B8']  - 1] / SCALE   # NIR
B11 = reprojected_bands[BAND_IDX['B11'] - 1] / SCALE   # SWIR (resampled)

print("  All bands extracted and scaled to reflectance [0–1]")

Step 2: Extracting bands (B11 confirmed at 10m via bilinear) ...
  All bands extracted and scaled to reflectance [0–1]


In [ ]:
# 3. CALCULATE SPECTRAL INDICES
print("Step 3: Calculating spectral indices ...")

# Safe division helper — avoids divide-by-zero
def safe_divide(a, b):
    with np.errstate(invalid='ignore', divide='ignore'):
        result = np.where(b != 0, a / b, np.nan)
    return result.astype(np.float32)

# --- Vegetation ---
# NDVI: Normalized Difference Vegetation Index (Captures green vegetation; high = dense forest, low = bare/sparse)
NDVI = safe_divide(B8 - B4, B8 + B4)
print("  ✓ NDVI")

# EVI: Enhanced Vegetation Index (Better than NDVI in dense forest (less saturation))
EVI = 2.5 * safe_divide(B8 - B4, B8 + 6 * B4 - 7.5 * B2 + 1)
EVI = np.clip(EVI, -1, 1).astype(np.float32)
print("  ✓ EVI")

# SAVI: Soil Adjusted Vegetation Index (L=0.5) (Separates sparse vegetation from bare soil)
SAVI = safe_divide(1.5 * (B8 - B4), B8 + B4 + 0.5)
print("  ✓ SAVI")

# --- Water / Moisture ---
# NDWI: Normalized Difference Water Index (Detects water bodies and soil moisture)
NDWI = safe_divide(B3 - B8, B3 + B8)
print("  ✓ NDWI")

# --- Bare Soil / Mines ---
# BSI: Bare Soil Index (Directly targets bare soil and disturbed land (mine pits, tailings))
BSI = safe_divide((B11 + B4) - (B8 + B2), (B11 + B4) + (B8 + B2))
print("  ✓ BSI")

# MBI: Modified Bare soil Index (More sensitive to mines vs natural bare soil than BSI)
MBI = safe_divide(B11 + B4 - B8, B11 + B4 + B8)
print("  ✓ MBI")

# NDBI: Normalized Difference Built-up Index (Captures disturbed/compacted surfaces like mine pits)
NDBI = safe_divide(B11 - B8, B11 + B8)       
print("  ✓ NDBI")

# --- Geology / Minerals ---
# CMI: Clay Mineral Index (Excellent for mines — picks up clay-rich tailings from rare earth extraction)
CMI = safe_divide(B11, B8)                  
print("  ✓ CMI")

# FCI: Ferrous/Iron Oxide Index (Mine tailings are often iron-rich — very discriminating for mine detection)
FCI = safe_divide(B11, B8)
print("  ✓ FCI")
print("Note: true ferrous index uses B11/B8A; B8 is used here as B8A not exported")

Step 3: Calculating spectral indices ...
  ✓ NDVI
  ✓ EVI
  ✓ SAVI
  ✓ NDWI
  ✓ BSI
  ✓ MBI
  ✓ NDBI
  ✓ CMI
  ✓ FCI


In [ ]:
# 4. STACK bands + indices and SAVE
print("Step 4: Stacking and saving output GeoTIFF ...")

band_data = [B2, B3, B4, B8, B11, NDVI, EVI, SAVI, NDWI, BSI, MBI, NDBI, CMI, FCI]
band_names = ['B2_Blue', 'B3_Green', 'B4_Red', 'B8_NIR', 'B11_SWIR_10m',
              'NDVI', 'EVI', 'SAVI', 'NDWI', 'BSI', 'MBI', 'NDBI', 'CMI', 'FCI']  #setting the band name

output_profile = reprojected_profile.copy()
output_profile.update({
    'count' : len(band_data),
    'dtype' : 'float32',
    'nodata': np.nan,
})

with rasterio.open(OUTPUT_TIF, 'w', **output_profile) as dst:
    for i, (data, name) in enumerate(zip(band_data, band_names), start=1):
        dst.write(data, i)
        dst.update_tags(i, name=name)

print(f"\n Done! Output saved to:\n  {OUTPUT_TIF}")

print(f"\nBand order in output file:")
for i, name in enumerate(band_names, start=1):
    print(f"  Band {i:>2}: {name}")


Step 4: Stacking and saving output GeoTIFF ...

✓ Done! Output saved to:
  Sentinel2_2025_Kachin_processed.tif

Band order in output file:
  Band  1: B2_Blue
  Band  2: B3_Green
  Band  3: B4_Red
  Band  4: B8_NIR
  Band  5: B11_SWIR_10m
  Band  6: NDVI
  Band  7: EVI
  Band  8: SAVI
  Band  9: NDWI
  Band 10: BSI
  Band 11: MBI
  Band 12: NDBI
  Band 13: CMI
  Band 14: FCI
